In [1]:
import os

# ---- Runtime crash fix (TF 2.18 + protobuf 6) ----
# Must be set before importing tensorflow (and before protobuf gets imported indirectly).
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd

# Import tensorflow only after env vars are set.
import tensorflow as tf

print("TF:", tf.__version__)
print("Num GPUs:", len(tf.config.list_physical_devices("GPU")))

try:
    tf.config.optimizer.set_jit(True)
except Exception as e:
    print("Could not enable XLA JIT:", repr(e))




2026-05-14 03:15:22.547295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778728522.559475      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778728522.563290      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF: 2.18.0
Num GPUs: 1


In [2]:
def auto_select_accelerator():
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.experimental.TPUStrategy(tpu)
        print("Running on TPU:", tpu.master())
    except Exception:
        strategy = tf.distribute.get_strategy()
    print(f"Running on {strategy.num_replicas_in_sync} replicas")
    return strategy


def build_decoder(with_labels=True, target_size=(300, 300), ext="jpg"):
    """
    Correctness-preserving: set static shape after resize to avoid retracing/shape overhead.
    """
    target_h, target_w = int(target_size[0]), int(target_size[1])

    def decode(path):
        file_bytes = tf.io.read_file(path)
        if ext == "png":
            img = tf.image.decode_png(file_bytes, channels=3)
        elif ext in ["jpg", "jpeg"]:
            img = tf.image.decode_jpeg(file_bytes, channels=3)
        else:
            raise ValueError("Image extension not supported")

        img = tf.cast(img, tf.float32) / 255.0
        img = tf.image.resize(img, (target_h, target_w))
        img = tf.ensure_shape(img, (target_h, target_w, 3))
        return img

    def decode_with_labels(path, label):
        return decode(path), label

    return decode_with_labels if with_labels else decode


def build_augmenter(with_labels=True, seed=(123, 456)):
    seed = tf.constant(seed, dtype=tf.int32)

    def augment(img):
        s = tf.stack([seed[0] + tf.shape(img)[0], seed[1] + tf.shape(img)[1]])
        img = tf.image.stateless_random_flip_left_right(img, seed=s)
        img = tf.image.stateless_random_flip_up_down(
            img, seed=s + tf.constant([1, 1], tf.int32)
        )
        return img

    def augment_with_labels(img, label):
        return augment(img), label

    return augment_with_labels if with_labels else augment


def build_dataset(
    paths,
    labels=None,
    bsize=32,
    cache=False,
    decode_fn=None,
    augment_fn=None,
    augment=True,
    repeat=True,
    shuffle=1024,
    cache_dir="",
):
    """
    Correctness-preserving runtime tuning:
    - keep deterministic ordering/augment behavior
    - use parallel map + prefetch
    - avoid slow cache-to-disk unless explicitly requested
    """
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter(labels is not None)

    AUTO = tf.data.AUTOTUNE
    slices = paths if labels is None else (paths, labels)
    dset = tf.data.Dataset.from_tensor_slices(slices)

    options = tf.data.Options()
    options.experimental_deterministic = True
    options.experimental_optimization.apply_default_optimizations = True
    options.experimental_optimization.map_parallelization = True
    options.experimental_optimization.parallel_batch = True
    dset = dset.with_options(options)

    if shuffle:
        dset = dset.shuffle(shuffle, reshuffle_each_iteration=True)
    if repeat:
        dset = dset.repeat()

    dset = dset.map(decode_fn, num_parallel_calls=AUTO, deterministic=True)

    if cache:
        if cache_dir:
            os.makedirs(os.path.dirname(cache_dir), exist_ok=True)
            dset = dset.cache(cache_dir)
        else:
            dset = dset.cache()

    if augment:
        dset = dset.map(augment_fn, num_parallel_calls=AUTO, deterministic=True)

    dset = dset.batch(bsize, drop_remainder=False)
    dset = dset.prefetch(AUTO)
    return dset




In [3]:
COMPETITION_NAME = "ranzcr-clip-catheter-line-classification"
load_dir = f"/kaggle/input/{COMPETITION_NAME}/"

strategy = auto_select_accelerator()
BATCH_SIZE = strategy.num_replicas_in_sync * 16

train_df = pd.read_csv(load_dir + "train.csv")
label_cols = [c for c in train_df.columns if c not in ["StudyInstanceUID", "PatientID"]]
NUM_LABELS = len(label_cols)

print("NUM_LABELS:", NUM_LABELS)
print("Label columns:", label_cols)

# Submission correctness fix:
# Some provided sample_submission.csv files in this environment have missing label columns.
# We build the submission frame using the sample's StudyInstanceUIDs, then reindex to ALL label columns.
sub_base = pd.read_csv(load_dir + "sample_submission.csv")
sub_df = pd.DataFrame({"StudyInstanceUID": sub_base["StudyInstanceUID"].values})
sub_df = sub_df.reindex(columns=["StudyInstanceUID"] + label_cols, fill_value=0.0)

test_paths = (load_dir + "test/" + sub_df["StudyInstanceUID"].values + ".jpg").astype(
    str
)
test_paths = tf.constant(test_paths)

IMSIZE = (224, 240, 260, 300, 380, 456, 528, 600)
target_size = (IMSIZE[3], IMSIZE[3])

test_decoder = build_decoder(with_labels=False, target_size=target_size)

dtest = build_dataset(
    test_paths,
    bsize=BATCH_SIZE,
    repeat=False,
    shuffle=False,
    augment=False,
    cache=False,
    decode_fn=test_decoder,
)




Running on 1 replicas
NUM_LABELS: 11
Label columns: ['ETT - Abnormal', 'ETT - Borderline', 'ETT - Normal', 'NGT - Abnormal', 'NGT - Borderline', 'NGT - Incompletely Imaged', 'NGT - Normal', 'CVC - Abnormal', 'CVC - Borderline', 'CVC - Normal', 'Swan Ganz Catheter Present']


I0000 00:00:1778728529.464839      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


In [4]:
def build_model(input_shape=(300, 300, 3), num_labels=11):
    base = tf.keras.applications.VGG16(
        include_top=False, weights="imagenet", input_shape=input_shape
    )
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(512, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(num_labels, activation="sigmoid")(x)
    model = tf.keras.Model(inputs=base.input, outputs=out)
    return model


with strategy.scope():
    model = build_model(
        input_shape=(target_size[0], target_size[1], 3), num_labels=NUM_LABELS
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        jit_compile=True,
    )

model.summary()



58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 300, 300, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 300, 300, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 150, 150, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 150, 150, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 150, 150, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 75, 75, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 75, 75, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 75, 75, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 75, 75, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 37, 37, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 37, 37, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 37, 37, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 37, 37, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 18, 18, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         5,643 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,982,987 (57.16 MB)

 Trainable params: 14,982,987 (57.16 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
tf.random.set_seed(42)
np.random.seed(42)

train_paths_all = (
    load_dir + "train/" + train_df["StudyInstanceUID"].values + ".jpg"
).astype(str)
train_labels = train_df[label_cols].astype("float32").values

idx = np.arange(len(train_df))
np.random.shuffle(idx)
split = int(len(idx) * 0.95)
tr_idx, va_idx = idx[:split], idx[split:]

train_paths_tr = tf.constant(train_paths_all[tr_idx])
train_labels_tr = tf.constant(train_labels[tr_idx])
train_paths_va = tf.constant(train_paths_all[va_idx])
train_labels_va = tf.constant(train_labels[va_idx])

train_decoder = build_decoder(with_labels=True, target_size=target_size)
train_augmenter = build_augmenter(with_labels=True)

dtrain = build_dataset(
    train_paths_tr,
    labels=train_labels_tr,
    bsize=BATCH_SIZE,
    repeat=True,
    shuffle=4096,
    augment=True,
    cache=False,
    decode_fn=train_decoder,
    augment_fn=train_augmenter,
)

valid_decoder = build_decoder(with_labels=True, target_size=target_size)

dvalid = build_dataset(
    train_paths_va,
    labels=train_labels_va,
    bsize=BATCH_SIZE,
    repeat=False,
    shuffle=False,
    augment=False,
    cache=False,
    decode_fn=valid_decoder,
)

steps_per_epoch = max(1, len(tr_idx) // BATCH_SIZE)
val_steps = max(1, int(np.ceil(len(va_idx) / BATCH_SIZE)))

history = model.fit(
    dtrain,
    validation_data=dvalid,
    epochs=1,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    verbose=1,
)



I0000 00:00:1778728534.089150     106 service.cc:148] XLA service 0x7f7eec0053d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778728534.089181     106 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1778728534.095505     106 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778728534.123192     106 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1607/1607 ━━━━━━━━━━━━━━━━━━━━ 447s 267ms/step - loss: 0.2872 - val_loss: 0.2757


In [6]:
pred = model.predict(dtest, verbose=1)
pred = np.clip(pred, 0.0, 1.0)

if pred.shape[0] != len(sub_df):
    raise RuntimeError(
        f"Prediction rows ({pred.shape[0]}) != submission rows ({len(sub_df)})."
    )
if pred.shape[1] != len(label_cols):
    raise RuntimeError(
        f"Prediction cols ({pred.shape[1]}) != num labels ({len(label_cols)})."
    )

sub_df.loc[:, label_cols] = pred.astype(np.float32)
sub_df.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", sub_df.shape)
print("Columns:", list(sub_df.columns))
print(sub_df.head())

189/189 ━━━━━━━━━━━━━━━━━━━━ 47s 239ms/step
Wrote submission.csv with shape: (3009, 12)
Columns: ['StudyInstanceUID', 'ETT - Abnormal', 'ETT - Borderline', 'ETT - Normal', 'NGT - Abnormal', 'NGT - Borderline', 'NGT - Incompletely Imaged', 'NGT - Normal', 'CVC - Abnormal', 'CVC - Borderline', 'CVC - Normal', 'Swan Ganz Catheter Present']
                                    StudyInstanceUID  ETT - Abnormal  \
0  1.2.826.0.1.3680043.8.498.25512976433640891933...        0.001373   
1  1.2.826.0.1.3680043.8.498.24449897997512078380...        0.006333   
2  1.2.826.0.1.3680043.8.498.38485493636649999035...        0.003667   
3  1.2.826.0.1.3680043.8.498.11901773728604504629...        0.009453   
4  1.2.826.0.1.3680043.8.498.11327663178389439022...        0.008974   

   ETT - Borderline  ETT - Normal  NGT - Abnormal  NGT - Borderline  \
0          0.003765      0.024568        0.004039          0.007708   
1          0.027895      0.297805        0.013026          0.019777   
2          0.02